## 实验 0：验证课程训练引擎的基本契约

**实验目的**：用 12 个合成样本走通一次训练、学习率调度和验证，确认 `common.engine` 返回的指标覆盖全部样本。

`train_one_epoch` 通过非空 optimizer 进入训练分支：设置训练模式，把 batch 移到 device，再依次清梯度、前向、计算 loss、反向和更新。断言确认三个 batch 的 12 个样本全部计入结果。

`StepLR` 在完整 epoch 后调用一次，改变后续 epoch 的学习率；不同 scheduler 的调用契约可能不同。`evaluate()` 内部已设置评估模式并进入 `inference_mode()`，所以外层同名上下文是冗余但无害的。loss 是按样本数加权的均值。

**边界提醒**：这里训练和验证使用同一批数据，只用于检查代码契约，不能估计泛化能力。

In [ ]:
import torch
from torch import nn
from torch.utils.data import DataLoader, TensorDataset
from common.engine import train_one_epoch, evaluate
from common.checkpoint import save_checkpoint, load_checkpoint
inputs = torch.randn(12, 4)
targets = (inputs[:, 0] > 0).long()
train_loader = DataLoader(TensorDataset(inputs, targets), batch_size=4)
validation_loader = DataLoader(TensorDataset(inputs, targets), batch_size=4)
model = nn.Sequential(nn.Linear(4, 8), nn.ReLU(), nn.Linear(8, 2))
loss_fn = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.01)
scheduler = torch.optim.lr_scheduler.StepLR(optimizer, step_size=1)
result = train_one_epoch(model, train_loader, loss_fn, optimizer, torch.device('cpu'))
scheduler.step()  # 该 scheduler 按 epoch 调用
assert result.examples == len(train_loader.dataset)
with torch.inference_mode():
    validation = evaluate(model, validation_loader, loss_fn, torch.device('cpu'))
assert torch.isfinite(torch.tensor(validation.loss))
print('weighted train loss:', result.loss, 'validation:', validation)

# 完整训练、验证与恢复训练

## 学习目标

能够解释一个 epoch 的执行顺序、训练与验证的状态边界、按样本聚合指标、早停策略，以及保存和恢复训练所需的状态。


## 概念模型与执行路径

训练循环负责产生梯度并更新参数；验证循环只测量当前参数在未参与更新的数据上的表现。一次训练 batch 的路径是：数据移到 device → 清梯度 → 前向 → 损失 → 反向 → 参数更新。epoch 结束后聚合指标并执行模型选择。最终测试应重新加载验证指标最好的 checkpoint，而不是默认使用最后一个 epoch。


### 实验 1：定位课程训练组件

**实验目的**：从不同 notebook 启动目录定位包含 `common` 包的课程根目录，确保后续能够导入训练引擎、运行时工具、早停和 checkpoint 模块。

代码检查当前目录、父目录和 `07-deep-learning/pytorch` 子目录，选择第一个包含 `common` 的候选项，再将其插入 `sys.path`。先判断路径是否存在，可以避免重复运行后出现相同条目。

**执行顺序注意**：实验 0 已提前导入 `common`，因此完整 notebook 仍应从可解析课程包的项目环境启动。正式项目更适合安装包或使用固定入口，而不是动态修改 `sys.path`。

In [ ]:
from pathlib import Path
import sys

candidates = [
    Path.cwd(),
    Path.cwd().parent,
    Path.cwd() / "07-deep-learning/pytorch",
]
PYTORCH_ROOT = next(path for path in candidates if (path / "common").exists())
if str(PYTORCH_ROOT) not in sys.path:
    sys.path.insert(0, str(PYTORCH_ROOT))
print("course root:", PYTORCH_ROOT)


### 实验 2：准备可复现的二分类训练状态

**实验目的**：建立后续训练所需的合成数据、互斥划分、DataLoader、模型、优化器和损失函数。

`seed_everything(42)` 设置 Python、NumPy 和 PyTorch 随机种子，使合成输入、模型初始化和当前单进程 shuffle 可复现；它并不自动保证所有硬件算子逐位确定。标签由前两个特征之和是否大于 0 得到，因此存在可学习的线性边界。

前 96 个样本训练，后 32 个验证；训练 loader shuffle，验证 loader 保持稳定顺序。模型输出两个 logits，与二分类 `CrossEntropyLoss` 匹配。Adam 还会维护一阶、二阶矩和 step 计数；这些状态要到第一次更新后才创建，也是恢复训练时必须保存的内容。

**观察重点**：本单元只构造状态，不执行训练。

In [ ]:
import tempfile
import torch
from torch import nn
from torch.utils.data import DataLoader, TensorDataset
from common.engine import train_one_epoch, evaluate
from common.runtime import seed_everything

seed_everything(42)
inputs = torch.randn(128, 4)
labels = (inputs[:, 0] + inputs[:, 1] > 0).long()
train_loader = DataLoader(TensorDataset(inputs[:96], labels[:96]), batch_size=16, shuffle=True)
validation_loader = DataLoader(TensorDataset(inputs[96:], labels[96:]), batch_size=16)
model = nn.Sequential(nn.Linear(4, 8), nn.ReLU(), nn.Linear(8, 2))
optimizer = torch.optim.Adam(model.parameters(), lr=0.03)
loss_fn = nn.CrossEntropyLoss()


### 实验 3：训练、验证并应用早停策略

**实验目的**：观察 epoch 级控制流，并用验证准确率决定是否刷新最佳记录或提前停止。每个 epoch 先训练，再用更新后的模型验证。

训练和验证都按样本累计指标。若 batch 平均损失为 $L_b$、样本数为 $n_b$，epoch loss 是 $\sum_b L_bn_b / \sum_b n_b$，避免较小的尾 batch 被过度加权。accuracy 也通过总正确数除以总样本数计算。

`EarlyStopping(patience=2)` 默认为最大化指标。第一次一定算改善；只有严格超过历史最佳值才重置计数，相等也算未改善。连续两个 epoch 未改善就停止。循环可能不足 5 次，后续 `epoch` 表示实际结束的最后一轮。

**重要限制**：该 EarlyStopping 只记录最佳值和陈旧轮数，不会保存或恢复最佳权重。生产训练应在 `improved=True` 时保存 best checkpoint；本页实验 4 保存的是循环结束时的当前模型，不保证是最佳模型。

In [ ]:
from common.training import EarlyStopping
stopping = EarlyStopping(patience=2)
for epoch in range(1, 6):
    train_result = train_one_epoch(model, train_loader, loss_fn, optimizer, torch.device("cpu"))
    validation_result = evaluate(model, validation_loader, loss_fn, torch.device("cpu"))
    improved, should_stop = stopping.update(validation_result.accuracy)
    print(epoch, train_result, validation_result, "improved=", improved)
    if should_stop:
        break


### 实验 4：保存并恢复模型与优化器状态

**实验目的**：验证 checkpoint 能重建模型预测，并携带恢复训练所需的 optimizer 状态、epoch 和指标。临时目录在 `with` 结束时自动删除，不会留下文件。

保存内容包括模型 `state_dict`、Adam state、epoch 和验证准确率。加载前先创建结构一致的新模型和同类型 optimizer。默认 `map_location='cpu'`，实现使用 `weights_only=True`，减少反序列化任意 Python 对象的风险。

断言比较两个模型对同一输入的 logits，证明参数恢复正确。Adam 的统计量不影响这次前向，却会影响下一次更新；通常从 `metadata['epoch'] + 1` 继续训练。若还使用 scheduler 或混合精度 scaler，也应一并保存和恢复。

**完整恢复边界**：要精确复现下一个 shuffle 或随机增强，还需保存随机数生成器状态；当前 helper 没有覆盖这一项。

In [ ]:
from common.checkpoint import save_checkpoint, load_checkpoint
with tempfile.TemporaryDirectory() as directory:
    path = save_checkpoint(Path(directory) / "resume.pt", model, optimizer, epoch=epoch,
                           metrics={"validation_accuracy": validation_result.accuracy})
    restored = nn.Sequential(nn.Linear(4, 8), nn.ReLU(), nn.Linear(8, 2))
    restored_optimizer = torch.optim.Adam(restored.parameters(), lr=0.03)
    metadata = load_checkpoint(path, restored, restored_optimizer)
    torch.testing.assert_close(model(inputs[:4]), restored(inputs[:4]))
    print("restored metadata:", metadata)


## 底层机制

`model.train()`/`eval()` 与梯度开关是两个独立维度：前者修改模块的 `training` 标志，影响 Dropout 和 BatchNorm；后者由 `enable_grad()`、`no_grad()` 或 `inference_mode()` 控制。验证通常需要评估模式加 `inference_mode()`。

课程引擎通过 optimizer 是否为 `None` 统一训练和验证路径。checkpoint 则是训练状态快照，不等于最佳模型策略：权重决定预测，optimizer 保存动量/自适应统计，scheduler 保存学习率进度，scaler 保存混合精度缩放状态，epoch 和指标决定控制流从哪里继续。缺一项可能仍可运行，但不再等价于无中断训练。

## 检查点

不看上文，先回答再验证：

1. 一个训练 batch 从数据搬运到参数更新的完整顺序是什么？
2. 为什么 epoch loss 要乘 batch size 后累计？
3. `evaluate()` 为什么同时需要评估模式和禁用梯度？
4. patience 为 2，准确率依次为 `0.70, 0.72, 0.72, 0.71` 时在哪轮停止？
5. 为什么测试应加载最佳验证 checkpoint？为什么不能用测试集选择最佳 epoch？
6. 只恢复权重、不恢复 Adam 状态，下一步更新是否与未中断训练相同？

## 试一试

1. 给模型加入 Dropout，在训练模式下对同一验证集评估三次，再切换评估模式比较波动。
2. 构造大小不同的两个 batch，比较按样本加权 loss 与简单平均 batch loss。
3. 在实验 3 中仅当 `improved=True` 时保存 checkpoint，训练结束后加载并验证保存指标等于 `stopping.best`。
4. 比较“连续训练两轮”和“训练一轮、保存恢复、再训练一轮”的最终权重，并控制第二轮采样顺序。
5. 加入 `StepLR`，将 scheduler 一并保存，恢复后比较 learning rate 和 `last_epoch`。

## 常见错误与调试

- **训练顺序错误**：漏掉清梯度会意外累积；使用清梯度 → 前向 → loss → 反向 → 更新作为清晰基线。
- **验证仍处于训练模式**：Dropout 随机、BatchNorm 更新统计。验证前调用 `model.eval()`。
- **验证记录计算图或更新参数**：浪费内存并污染评估。使用 `inference_mode()`，验证路径不传 optimizer。
- **简单平均不等长 batch**：尾 batch 被过度加权。按实际样本数累计。
- **只保存最后模型**：早停不会自动回退。验证改善时保存，测试前加载 best checkpoint。
- **用测试集选择模型**：测试信息泄漏进开发过程。只用验证集选择，测试集最后使用。
- **恢复时丢弃 optimizer/scheduler/scaler**：训练轨迹跳变。恢复所有影响下一步更新的状态。
- **epoch off-by-one**：若保存的是已完成轮次，恢复通常从 `epoch + 1` 开始。
- **空 DataLoader**：课程引擎会抛出 `ValueError`；检查划分和 `drop_last`。